# 04 · LLM + Propose-Verify-Repair (GPU)
Add inputs: LRS3 trainval shards, lrs3-test, voxmm-test, and the output of notebook 03 (`best.pt`); for Track B also run `git clone https://github.com/mpc001/auto_avsr /kaggle/working/auto_avsr` first. Accelerator **GPU T4**, Internet **On**, secret **HF_TOKEN** (with Llama 3.2 access).

Without Llama access set `LLM = 'Qwen/Qwen2.5-1.5B'`.

In [ ]:
# --- setup: clone the project code ---
import os, subprocess
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/durgaanuhya02/PVR-Lip-Reading.git", "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
!git log --oneline -1

In [ ]:
!pip install -q cmudict peft bitsandbytes

In [ ]:
# --- load a Kaggle secret into an environment variable (value is never printed) ---
def load_secret(name):
    from kaggle_secrets import UserSecretsClient
    os.environ[name] = UserSecretsClient().get_secret(name)
    print(f"{name} loaded")
load_secret('HF_TOKEN')

In [ ]:
import glob
LRS3_TRAIN = sorted(glob.glob('/kaggle/input/**/lrs3_trainval_shard*/index.jsonl', recursive=True))
LRS3_TEST = glob.glob('/kaggle/input/**/lrs3_test_rois/index.jsonl', recursive=True)[0]
VOXMM_TEST = sorted(glob.glob('/kaggle/input/**/voxmm_test_shard*/index.jsonl', recursive=True))
TRACK = 'vit'   # which notebook-03 model to use: 'vit' (Track A) or 'autoavsr' (Track B)
VISUAL = glob.glob(f'/kaggle/input/**/visual_lrs3_{TRACK}/best.pt', recursive=True)[0]
LLM = 'meta-llama/Llama-3.2-1B'
TRAIN_ARGS, VOXMM_ARGS = ' '.join(LRS3_TRAIN), ' '.join(VOXMM_TEST)
print(LRS3_TRAIN, LRS3_TEST, VOXMM_TEST, VISUAL, sep='\n')

## Stage 2 · phoneme→sentence corpus (LRS3 train transcripts + WikiText, as in VALLR) and LoRA

In [ ]:
!cat {TRAIN_ARGS} > /kaggle/working/lrs3_train_all.jsonl
!python -m src.llm.build_corpus --manifest /kaggle/working/lrs3_train_all.jsonl --split train --wikitext 50000 --out /kaggle/working/llm_corpus.jsonl

In [ ]:
!python -m src.llm.train_p2t --config configs/llm_p2t.yaml --set model_name={LLM} max_steps=3000

## Stage 3 · tune λ on LRS3 validation, then test on LRS3 and LRS-VoxMM

In [ ]:
ADAPTER = '/kaggle/working/runs/llm_p2t/adapter'
!python -m src.pvr.run --visual {VISUAL} --index {TRAIN_ARGS} --split val --llm {LLM} --adapter {ADAPTER} --out /kaggle/working/runs/pvr_lrs3_val --n 5 --limit 300 --sweep | tail -30

In [ ]:
import json
sweep = json.load(open('/kaggle/working/runs/pvr_lrs3_val/metrics.json'))['lambda_sweep']
LAM = float(min(sweep, key=sweep.get)); print('best lambda', LAM, sweep)

In [ ]:
!python -m src.pvr.run --visual {VISUAL} --index {LRS3_TEST} --split test --llm {LLM} --adapter {ADAPTER} --out /kaggle/working/runs/pvr_lrs3_test --n 5 --lam {LAM} --sweep | tail -40

In [ ]:
!python -m src.pvr.run --visual {VISUAL} --index {VOXMM_ARGS} --split test --llm {LLM} --adapter {ADAPTER} --out /kaggle/working/runs/pvr_voxmm_test --n 5 --lam {LAM} --sweep | tail -40